# Normalization

- The normalization step involves some general cleanup, such as removing needless whitespace, lowercasing, and/or removing accents.
- If you’re familiar with [Unicode normalization](http://www.unicode.org/reports/tr15/) (such as NFC or NFKC), this is also something the tokenizer may apply.
- The 🤗 Transformers `tokenizer` has an attribute called `backend_tokenizer` that provides access to the underlying tokenizer from the 🤗 Tokenizers library
- The `normalizer` attribute of the `tokenizer` object has a `normalize_str()` method that we can use to see how the normalization is performed.
- In the following example, since we picked the `bert-base-uncased` checkpoint, the normalization applies lowercasing and removes the accents.

In [ ]:
from transformers import AutoTokenizer
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
print(type(bert_tokenizer.backend_tokenizer))
print(bert_tokenizer.backend_tokenizer.normalizer.normalize_str("Héllò hôw are ü?"))

# Pre-tokenization

- A tokenizer cannot be trained on raw text alone.
- Instead, we first need to split the texts into small entities, like words. That’s where the pre-tokenization step comes in.
- a word-based tokenizer can simply split a raw text into words on **whitespace and punctuation**.
- Those words will be the boundaries of the subtokens the tokenizer can learn during its training.
- To see how a fast tokenizer performs pre-tokenization, we can use the `pre_tokenize_str()` method of the `pre_tokenizer` attribute of the `tokenizer` object.
- Notice how the tokenizer is already keeping track of the offsets, which is how it can give us the offset mapping we used in the previous section.
- Here the **BERT** tokenizer **ignores the two spaces** and replaces them with just one, but the **offset jumps** between `are` and `you` to account for that.
- The BERT tokenizer uses splitting on whitespace and punctuation.

---
- **GPT** tokenizer will split on whitespace and punctuation as well
- but it will keep the spaces and replace them with a `Ġ` symbol, enabling it to recover the original spaces if we decode the tokens.
- Also note that unlike the BERT tokenizer, GPT tokenizer does not ignore the double space.

---
- The last example uses the **T5 tokenizer**, which is based on the **SentencePiece** algorithm.
- Like the GPT-2 tokenizer, this one keeps spaces and replaces them with a specific token (`_`)
- but the T5 tokenizer only splits on whitespace, not punctuation.
- Also note that it added a space by default at the beginning of the sentence (before `Hello`) and ignored the double space between `are` and `you`.

In [ ]:
bert_pre_tokenzied_str = bert_tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str("Hello, how are  you?")
print(bert_pre_tokenzied_str)
print("=====================")
gpt_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt_pre_tokenzied_str = gpt_tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str("Hello, how are  you?")
print(gpt_pre_tokenzied_str)
print("=====================")

t5_tokenizer = AutoTokenizer.from_pretrained("t5-small")
t5_pre_tokenzied_str = t5_tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str("Hello, how are  you?")
print(t5_pre_tokenzied_str)
print("=====================")

# SentencePiece

- [SentencePiece](https://github.com/google/sentencepiece) is a tokenization algorithm for the preprocessing of text
- You can use it with any of the models we will see in the next sections (**BPE, WordPiece, Unigram**).
- It considers the text as a sequence of Unicode characters, and replaces spaces with a special character, `▁`.
- Used in conjunction with the [Unigram](https://huggingface.co/course/chapter6/7) algorithm, it doesn’t even require a pre-tokenization step, which is very useful for languages where the space character is not used (like Chinese or Japanese).
- The other main feature of SentencePiece is __reversible tokenization__: since there is no special treatment of spaces, decoding the tokens is done simply by concatenating them and replacing the `_`s with spaces — this results in the normalized text.
    - As we saw earlier, the BERT tokenizer removes repeating spaces, so its tokenization is not reversible.